In [ ]:
# ============================================================
# EXPERIMENT 4A : KNN CLASSIFICATION FROM SCRATCH
# ============================================================
import numpy as np
from collections import Counter

class KNNClassifier:
    def __init__(self, k=3):
        self.k = k
        self.X_train = None
        self.y_train = None

    # Training
    def fit(self, X, y):
        self.X_train = np.array(X)
        self.y_train = np.array(y)

    # Euclidean distance
    def euclidean_distance(self, x1, x2):
        return np.sqrt(np.sum((x1 - x2) ** 2))

    # Predict one sample
    def predict_one(self, x):
        distances = []

        for i in range(len(self.X_train)):
            distance = self.euclidean_distance(x, self.X_train[i])
            distances.append((distance, self.y_train[i]))

        # Sort according to distance
        distances.sort(key=lambda item: item[0])

        # Select K nearest neighbors
        k_nearest = distances[:self.k]

        # Extract class labels
        neighbor_labels = [label for distance, label in k_nearest]

        # Majority voting
        prediction = Counter(neighbor_labels).most_common(1)[0][0]
        return prediction

    # Predict multiple samples
    def predict(self, X):
        X = np.array(X)
        predictions = []

        for x in X:
            predictions.append(self.predict_one(x))

        return np.array(predictions)

    # Accuracy
    def accuracy(self, y_true, y_pred):
        y_true = np.array(y_true)
        y_pred = np.array(y_pred)
        correct = np.sum(y_true == y_pred)
        return correct / len(y_true)

# Example Dataset
# X1 = Study Hours
# X2 = Attendance
# 0 = Fail
# 1 = Pass

X_train = np.array([
    [1, 60],
    [2, 65],
    [3, 70],
    [4, 75],
    [5, 80],
    [6, 85]
])

y_train = np.array([0, 0, 1, 1, 1, 1])

X_test = np.array([
    [2.5, 68],
    [4.5, 78],
    [5.5, 82]
])

y_test = np.array([0, 1, 1])

# Create model
knn = KNNClassifier(k=3)

# Train
knn.fit(X_train, y_train)

# Predict
y_pred = knn.predict(X_test)

# Display results
print("Actual Labels :", y_test)
print("Predicted Labels :", y_pred)

# Accuracy
accuracy = knn.accuracy(y_test, y_pred)
print("Accuracy :", accuracy)
print("Accuracy (%) :", accuracy * 100)

In [ ]:
# ============================================================
# EXPERIMENT 4B : KNN REGRESSION FROM SCRATCH
# ============================================================
import numpy as np

class KNNRegressor:
    def __init__(self, k=3):
        self.k = k
        self.X_train = None
        self.y_train = None

    # Training
    def fit(self, X, y):
        self.X_train = np.array(X, dtype=float)
        self.y_train = np.array(y, dtype=float)

    # Euclidean distance
    def euclidean_distance(self, x1, x2):
        return np.sqrt(np.sum((x1 - x2) ** 2))

    # Predict one sample
    def predict_one(self, x):
        distances = []

        for i in range(len(self.X_train)):
            distance = self.euclidean_distance(x, self.X_train[i])
            distances.append((distance, self.y_train[i]))

        # Sort according to distance
        distances.sort(key=lambda item: item[0])

        # Select K nearest neighbors
        k_nearest = distances[:self.k]

        # Extract target values
        neighbor_values = [value for distance, value in k_nearest]

        # Mean of target values
        prediction = np.mean(neighbor_values)
        return prediction

    # Predict multiple samples
    def predict(self, X):
        X = np.array(X, dtype=float)
        predictions = []

        for x in X:
            predictions.append(self.predict_one(x))

        return np.array(predictions)

    # Mean Squared Error
    def mse(self, y_true, y_pred):
        return np.mean((y_true - y_pred) ** 2)

    # Mean Absolute Error
    def mae(self, y_true, y_pred):
        return np.mean(np.abs(y_true - y_pred))

    # R2 Score
    def r2_score(self, y_true, y_pred):
        ss_res = np.sum((y_true - y_pred) ** 2)
        ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
        return 1 - (ss_res / ss_tot)

# Example Dataset
# X1 = House Area
# X2 = Number of Bedrooms
# Y = House Price in Lakhs

X_train = np.array([
    [800, 2],
    [1000, 2],
    [1200, 3],
    [1400, 3],
    [1600, 4],
    [1800, 4]
])

y_train = np.array([40, 50, 60, 70, 80, 90])

X_test = np.array([
    [1100, 2],
    [1500, 3],
    [1700, 4]
])

y_test = np.array([55, 75, 85])

# Create model
knn = KNNRegressor(k=3)

# Train
knn.fit(X_train, y_train)

# Predict
y_pred = knn.predict(X_test)

print("Actual Values :", y_test)
print("Predicted Values :", np.round(y_pred, 2))

# Evaluation
mse = knn.mse(y_test, y_pred)
mae = knn.mae(y_test, y_pred)
r2 = knn.r2_score(y_test, y_pred)

print("\nPerformance")
print("------------------")
print("MSE :", round(mse, 4))
print("MAE :", round(mae, 4))
print("R2 :", round(r2, 4))

In [ ]:
# ============================================================
# EXPERIMENT 4C : KNN CLASSIFICATION USING SCIKIT-LEARN
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

# --------------------------------------------
# Step 1: Load Dataset
# --------------------------------------------
iris = load_iris()

X = iris.data
y = iris.target

feature_names = iris.feature_names
target_names = iris.target_names

# --------------------------------------------
# Step 2: Create DataFrame
# --------------------------------------------
df = pd.DataFrame(X, columns=feature_names)

df["Target"] = y
df["Species"] = df["Target"].map({
    0: "setosa",
    1: "versicolor",
    2: "virginica"
})

print("Dataset Shape:")
print(df.shape)

print("\nFirst Five Records:")
print(df.head())

# --------------------------------------------
# Step 3: Dataset Information
# --------------------------------------------
print("\nFeature Names:")
print(feature_names)

print("\nTarget Classes:")
print(target_names)

print("\nClass Distribution:")
print(df["Species"].value_counts())

# --------------------------------------------
# Step 4: Train-Test Split
# --------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining Samples:", X_train.shape[0])
print("Testing Samples:", X_test.shape[0])

# --------------------------------------------
# Step 5: Feature Scaling
# --------------------------------------------
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --------------------------------------------
# Step 6: Create KNN Classifier
# --------------------------------------------
knn = KNeighborsClassifier(
    n_neighbors=5,
    weights="uniform",
    metric="euclidean"
)

# --------------------------------------------
# Step 7: Train Model
# --------------------------------------------
knn.fit(X_train_scaled, y_train)

# --------------------------------------------
# Step 8: Prediction
# --------------------------------------------
y_pred = knn.predict(X_test_scaled)

# --------------------------------------------
# Step 9: Actual vs Predicted
# --------------------------------------------
results = pd.DataFrame({
    "Actual": y_test,
    "Predicted": y_pred
})

results["Actual Species"] = results["Actual"].map(
    dict(enumerate(target_names))
)

results["Predicted Species"] = results["Predicted"].map(
    dict(enumerate(target_names))
)

print("\nActual vs Predicted:")
print(results)

# --------------------------------------------
# Step 10: Accuracy
# --------------------------------------------
accuracy = accuracy_score(y_test, y_pred)

print("\n=================================")
print("KNN CLASSIFICATION PERFORMANCE")
print("=================================")

print("Accuracy :", round(accuracy, 4))
print("Accuracy (%) :", round(accuracy * 100, 2))

# --------------------------------------------
# Step 11: Confusion Matrix
# --------------------------------------------
cm = confusion_matrix(y_test, y_pred)

print("\nConfusion Matrix:")
print(cm)

# --------------------------------------------
# Step 12: Classification Report
# --------------------------------------------
print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=target_names
))

# --------------------------------------------
# Step 13: Confusion Matrix Plot
# --------------------------------------------
plt.figure(figsize=(6, 5))

plt.imshow(cm, interpolation="nearest")

plt.title("KNN Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")

plt.xticks(range(len(target_names)), target_names)
plt.yticks(range(len(target_names)), target_names)

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.colorbar()
plt.tight_layout()
plt.show()